# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anish494/flyrank_ai_first_assignment/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)


**Task type: Clustering.**

My lane (Structured Content Archetype Clustering) is not classification, because there is
no pre-existing label like "declining" or "not declining" that I'm trying to predict.
It's not ranking, because I'm not ordering pages by priority against a single score.
It's clustering: I'm grouping content items by similarity across several numeric
behavioral signals (impressions, CTR, position, freshness, word count), with no
predefined "correct" group — the groups themselves are what I'm trying to discover.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy


Clustering has no target column to predict — there's no observed outcome or ground-truth
label I'm trying to match, unlike classification's "is_declining" or a regression's
numeric outcome. Instead, the "output" I'm producing is a **cluster assignment**: an
integer label (0, 1, 2, ...) that the algorithm itself creates for each page, based on
how similar it is to other pages across my chosen features. This is a proxy in the sense
that the cluster ID is not measuring anything real by itself — it only becomes meaningful
after I inspect what's actually inside each cluster and name it based on its typical
behavior (e.g. "stale visible pages"). Until that naming step, the cluster ID alone means
nothing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric


I will use two complementary checks rather than one number, because clustering
"success" is not as clean-cut as a classification accuracy score:

1. **Silhouette score** — a standard clustering metric between -1 and 1, measuring how
   well-separated and internally consistent the clusters are. A score meaningfully above
   0 means pages within a cluster are more similar to each other than to pages in other
   clusters — a bare-minimum signal that the grouping isn't arbitrary.
2. **Cluster interpretability (a practical, human check)** — for each cluster, I will
   look at its typical values across my features and ask: can I describe this cluster in
   one sentence a content strategist would recognize (e.g. "high traffic, good position,
   low CTR")? If a cluster's profile is muddled or indistinguishable from another
   cluster's, that's a sign K is wrong or the features aren't capturing real structure —
   even if the silhouette score looks fine.

"Good" means: a silhouette score clearly above 0 AND every cluster has a describable,
distinct profile a human reviewer could act on.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe


One row = one content item (one page). Below, I load the starter dataset and select the
numeric behavioral features I'll use for clustering — the same columns whose spread I
showed in ML-02 (impressions, position) plus a couple more that round out a page's
behavioral profile.

In [6]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Anish494/flyrank_ai_first_assignment"
REPO_DIR = "flyrank_ai_first_assignment"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()

cluster_features = ["impressions_90d", "avg_position", "ctr", "word_count", "content_age_days"]
unit_of_analysis = df[["content_id"] + cluster_features].copy()

print("Shape (rows, columns):", unit_of_analysis.shape)
print("\nOne row = one content item. First 5 rows:")
unit_of_analysis.head()

Shape (rows, columns): (30000, 6)

One row = one content item. First 5 rows:


,content_id,impressions_90d,avg_position,ctr,word_count,content_age_days
0,content_304f48230142,3803,10.6,0.76,3221.0,187
1,content_a1fb4e703a9e,15320,20.3,0.05,2481.0,445
2,content_9aa793d4d895,12581,36.5,0.09,3515.0,141
3,content_331d6c4de07b,11751,6.2,0.49,NaN,463
4,content_d99b7a2d90ca,19140,44.0,0.13,2803.0,263


## 5. Why ML beats a fixed rule here

A fixed rule works when a human can write a small number of clean if/else thresholds
that capture the pattern. That breaks down here for a few real reasons:

1. **Multiple dimensions interact at once.** A page's "type" depends jointly on
   impressions, position, CTR, word count, and age — not any single one of them. A
   hand-written rule like "if impressions > 1000 and position < 10" only carves the
   data along axis-aligned boxes; it can't capture a page that's mediocre on every
   individual metric but forms a genuinely distinct combination (e.g. moderate traffic,
   moderate position, but unusually low CTR for that position — a specific, real
   archetype a single-metric rule would miss entirely).
2. **I don't know the right thresholds in advance.** ML-02 already showed
   impressions_90d spans from 1 to 517,715 and avg_position from 0 to 245 — I have no
   principled way to pick cutoffs like "1000 impressions = high" without first looking
   at how pages naturally group. Clustering finds the natural breakpoints in the data
   itself, rather than me guessing them upfront.
3. **The number of plausible archetypes isn't obvious ahead of time.** A fixed rule
   requires deciding the categories before looking at the data. Clustering lets the
   structure emerge first, and I name the archetypes only after inspecting what's
   actually inside each group — which matches the lane guide's explicit warning against
   naming clusters before looking at them.
4. **This is fundamentally an unsupervised problem.** There's no historical "correct
   archetype" label anywhere in this dataset to write a rule against or validate one
   with — the entire point of this lane is discovering groups that were never labeled
   in the first place, which by definition requires a method built for finding
   structure, not one built for matching pre-defined categories.

In [7]:
# Quick check: are these features actually giving independent information,
# or are some of them redundant with each other?
correlation_matrix = unit_of_analysis[cluster_features].corr().round(2)
print(correlation_matrix)

                  impressions_90d  avg_position   ctr  word_count  \
impressions_90d              1.00         -0.07 -0.02        0.16   
avg_position                -0.07          1.00 -0.07        0.12   
ctr                         -0.02         -0.07  1.00       -0.12   
word_count                   0.16          0.12 -0.12        1.00   
content_age_days            -0.00          0.16  0.01       -0.12   

                  content_age_days  
impressions_90d              -0.00  
avg_position                  0.16  
ctr                           0.01  
word_count                   -0.12  
content_age_days              1.00  


**Confirmed by the correlation matrix above:** all pairwise correlations between the
five features are small (-0.12 to 0.16) — none of them are redundant with each other.
This means each feature is contributing genuinely independent information about a
page's behavior, which is exactly why a rule based on one or two thresholds couldn't
capture the same structure that clustering across all five dimensions at once can.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.